# ISYE 7406 Project — Data Preparation

This notebook starts the **data preparation** stage for the bird species distribution project.

## Goals
- Load each species dataset
- Check dimensions, columns, class balance, and missing values
- Standardize column names for cleaner modeling
- Convert `land_cover` to a categorical feature
- Build a single combined dataframe for cross-species EDA
- Build a clean species-specific modeling dictionary for later modeling

**Files used**
- `hawk_env.csv`
- `meadowlark_env.csv`
- `robin_env.csv`
- `towhee_env.csv`

> Note: your proposal mentions five species, but only **four CSV files** are currently available in this workspace. So this notebook prepares the four uploaded species first.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

DATA_DIR = Path(".")  # assumes notebook is in same folder as the csv files

file_map = {
    "Red-tailed Hawk": "hawk_env.csv",
    "Western Meadowlark": "meadowlark_env.csv",
    "American Robin": "robin_env.csv",
    "Spotted Towhee": "towhee_env.csv",
}


## 1. Load the datasets

In [ ]:
dfs = {}

for species_name, filename in file_map.items():
    df = pd.read_csv(DATA_DIR / filename)
    dfs[species_name] = df
    print(f"{species_name}: {df.shape[0]} rows x {df.shape[1]} columns")


## 2. Quick structure check

In [ ]:
for species_name, df in dfs.items():
    print("=" * 80)
    print(species_name)
    print("- Columns:")
    print(df.columns.tolist())
    print("- First 3 rows:")
    display(df.head(3))


## 3. Data quality checks

In [ ]:
summary_rows = []

for species_name, df in dfs.items():
    summary_rows.append({
        "species": species_name,
        "rows": len(df),
        "columns": df.shape[1],
        "presence_count": int((df["presence"] == 1).sum()),
        "absence_count": int((df["presence"] == 0).sum()),
        "presence_rate": round(df["presence"].mean(), 4),
        "missing_values_total": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

quality_summary = pd.DataFrame(summary_rows)
quality_summary


### What to look for
For the write-up, this table helps you say whether:
- the datasets are reasonably balanced or imbalanced,
- there are any missing values,
- duplicates need to be removed,
- the species datasets are similar in size.


## 4. Rename the WorldClim variables to cleaner names

In [ ]:
bio_rename = {
    "wc2.1_2.5m_bio_1": "bio1_annual_mean_temp",
    "wc2.1_2.5m_bio_2": "bio2_mean_diurnal_range",
    "wc2.1_2.5m_bio_3": "bio3_isothermality",
    "wc2.1_2.5m_bio_4": "bio4_temp_seasonality",
    "wc2.1_2.5m_bio_5": "bio5_max_temp_warmest_month",
    "wc2.1_2.5m_bio_6": "bio6_min_temp_coldest_month",
    "wc2.1_2.5m_bio_7": "bio7_temp_annual_range",
    "wc2.1_2.5m_bio_8": "bio8_mean_temp_wettest_quarter",
    "wc2.1_2.5m_bio_9": "bio9_mean_temp_driest_quarter",
    "wc2.1_2.5m_bio_10": "bio10_mean_temp_warmest_quarter",
    "wc2.1_2.5m_bio_11": "bio11_mean_temp_coldest_quarter",
    "wc2.1_2.5m_bio_12": "bio12_annual_precip",
    "wc2.1_2.5m_bio_13": "bio13_precip_wettest_month",
    "wc2.1_2.5m_bio_14": "bio14_precip_driest_month",
    "wc2.1_2.5m_bio_15": "bio15_precip_seasonality",
    "wc2.1_2.5m_bio_16": "bio16_precip_wettest_quarter",
    "wc2.1_2.5m_bio_17": "bio17_precip_driest_quarter",
    "wc2.1_2.5m_bio_18": "bio18_precip_warmest_quarter",
    "wc2.1_2.5m_bio_19": "bio19_precip_coldest_quarter",
}


In [ ]:
clean_dfs = {}

for species_name, df in dfs.items():
    clean_df = df.rename(columns=bio_rename).copy()
    clean_df["land_cover"] = clean_df["land_cover"].astype("category")
    clean_dfs[species_name] = clean_df

clean_dfs["American Robin"].head()


## 5. Combine all species into one dataframe for EDA

In [ ]:
combined_df = pd.concat(clean_dfs.values(), ignore_index=True)

print(combined_df.shape)
combined_df.head()


In [ ]:
combined_df["species"].value_counts()


This combined dataframe is mainly useful for:
- cross-species EDA,
- comparing environmental ranges,
- checking how habitat variables differ across species.


## 6. Define predictor columns for later modeling

In [ ]:
target_col = "presence"

id_cols = ["longitude", "latitude", "species"]
categorical_cols = ["land_cover"]

feature_cols = [col for col in combined_df.columns if col not in [target_col] + id_cols]

numeric_feature_cols = [col for col in feature_cols if col not in categorical_cols]

print("Target:", target_col)
print("Categorical features:", categorical_cols)
print("Numeric feature count:", len(numeric_feature_cols))
print("Numeric features:")
print(numeric_feature_cols)


## 7. Optional: remove coordinates from the modeling feature set

For a species distribution project like this, you should think carefully about whether to use
`longitude` and `latitude` as predictors.

### Why you might exclude them
- They can let the model memorize geography instead of learning environmental relationships.
- Your proposal emphasizes prediction from **environmental variables**.

### Why you might include them later
- As a comparison model or sensitivity check.

For now, this notebook **excludes coordinates from the main feature set**.


## 8. Create species-specific modeling datasets

In [ ]:
model_data = {}

for species_name, df in clean_dfs.items():
    X = df[feature_cols].copy()
    y = df[target_col].copy()
    model_data[species_name] = {"X": X, "y": y, "full_df": df}

for species_name, parts in model_data.items():
    print(f"{species_name}: X shape = {parts['X'].shape}, y mean = {parts['y'].mean():.3f}")


## 9. Quick land cover sanity check

In [ ]:
for species_name, df in clean_dfs.items():
    print("=" * 80)
    print(species_name)
    display(df["land_cover"].value_counts().sort_index().rename("count").to_frame().head(15))


## 10. Save cleaned datasets (optional)

This is helpful so your later modeling notebooks start from a clean version.


In [ ]:
output_dir = Path("cleaned_data")
output_dir.mkdir(exist_ok=True)

for species_name, df in clean_dfs.items():
    safe_name = species_name.lower().replace(" ", "_").replace("-", "")
    df.to_csv(output_dir / f"{safe_name}_clean.csv", index=False)

combined_df.to_csv(output_dir / "all_species_combined_clean.csv", index=False)

print("Saved cleaned files to:", output_dir.resolve())


## 11. Suggested write-up text for the data prep section

You can adapt something like this for the report:

> We began by loading the species-specific environmental datasets for four bird species: American Robin, Red-tailed Hawk, Spotted Towhee, and Western Meadowlark. Each dataset contained a binary response variable indicating observed presence versus pseudo-absence, geographic coordinates, 19 WorldClim bioclimatic predictors, a MODIS land cover class, and the species label. We verified dataset dimensions, class balance, missing values, and duplicate rows before modeling. To improve interpretability, the WorldClim variable names were renamed from their original raster-style labels to readable bioclimatic names, and land cover was converted to a categorical variable. We then created both species-specific modeling datasets and a combined dataframe for cross-species exploratory analysis. Because the project focuses on prediction from environmental variables, longitude and latitude were excluded from the main feature set to avoid allowing models to rely directly on location rather than ecological covariates.


## Next step

After this notebook, the natural next notebook is:

**`02_eda.ipynb`**
- class balance plots
- histograms / density plots of key climate variables
- correlation heatmap
- land cover comparison across species
